# Análisis de sentimiento de reseñas

## Objetivo de negocio

WeLoveReviews necesita comprobar si el texto de 500 reseñas coincide con la puntuación media comunicada de **4.5/5**. Mediremos la distribución de sentimiento escrito, la compararemos con la puntuación humana y documentaremos los falsos negativos antes de entregar el reporte.

El modelo elegido es `nlptown/bert-base-multilingual-uncased-sentiment`, fijado a un commit concreto. Devuelve estrellas de 1 a 5, que agruparemos como **negative** (1-2), **neutral** (3) y **positive** (4-5).

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from app import DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH, MODEL_NAME, MODEL_REVISION, enrich_reviews

reviews = pd.read_csv(DEFAULT_INPUT_PATH)
print(f"Filas: {len(reviews)}")
print(f"Columnas: {reviews.columns.tolist()}")
print(f"Valores nulos totales: {reviews.isna().sum().sum()}")
print("Distribución de la puntuación humana:")
display(reviews["rating"].value_counts().sort_index().rename("count").to_frame())

Filas: 500
Columnas: ['review_id', 'rating', 'review_text']
Valores nulos totales: 0
Distribución de la puntuación humana:


,count
rating,
1,12
2,18
3,38
4,72
5,360


## EDA, limpieza y plan de acción

El archivo contiene 500 reseñas completas y las columnas necesarias para relacionar texto, puntuación humana e identificador. No se eliminan reseñas: el texto se convierte a cadena justo antes de inferir y se conservan las columnas originales para auditar cada resultado.

El modelo es adecuado como primera referencia porque entiende varios idiomas y produce una escala de cinco estrellas. Hay que interpretar sus resultados con cautela: fue ajustado con reseñas de productos, mientras que este conjunto describe servicios. Por eso prestaremos especial atención a reseñas con alta puntuación humana pero predicción de 1 o 2 estrellas.

In [2]:
enriched = enrich_reviews(DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH)

band_order = ["negative", "neutral", "positive"]
distribution = enriched["sentiment_band"].value_counts(normalize=True).reindex(band_order, fill_value=0)
summary = pd.DataFrame({
    "count": enriched["sentiment_band"].value_counts().reindex(band_order, fill_value=0),
    "percentage": (distribution * 100).round(1),
})
print(f"Modelo: {MODEL_NAME}")
print(f"Commit fijado: {MODEL_REVISION}")
print(f"Salida escrita en: {DEFAULT_OUTPUT_PATH}")
display(summary)

Device set to use cpu


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/home/vscode/.local/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/vscode/.local/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _l

Modelo: nlptown/bert-base-multilingual-uncased-sentiment
Commit fijado: 8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
Salida escrita en: /workspaces/thispedrito1-machine-learning-python-template/data/processed/reviews_with_sentiment.csv


,count,percentage
sentiment_band,,
negative,47,9.4
neutral,41,8.2
positive,412,82.4


## Resultados frente al promedio de 4.5 estrellas

La puntuación humana de referencia es 4.5/5. La distribución del modelo no pretende sustituir ese promedio: sirve para detectar si el lenguaje escrito refuerza la señal de las estrellas o revela una experiencia más matizada.

In [3]:
human_average = enriched["rating"].mean()
positive_share = (enriched["sentiment_band"] == "positive").mean() * 100
high_rating_share = (enriched["rating"] >= 4).mean() * 100
print(f"Promedio de puntuación humana: {human_average:.2f}/5 (referencia comunicada: 4.5/5)")
print(f"Reseñas positivas según el modelo: {positive_share:.1f}%")
print(f"Reseñas con 4-5 estrellas humanas: {high_rating_share:.1f}%")
print("Lectura: una media alta de estrellas puede coexistir con lenguaje neutral o negativo; por eso se revisan ambos indicadores.")

Promedio de puntuación humana: 4.50/5 (referencia comunicada: 4.5/5)
Reseñas positivas según el modelo: 82.4%
Reseñas con 4-5 estrellas humanas: 86.4%
Lectura: una media alta de estrellas puede coexistir con lenguaje neutral o negativo; por eso se revisan ambos indicadores.


## Falsos negativos y verificación manual

Consideramos falso negativo una reseña con 4 o 5 estrellas humanas que el modelo coloca en 1 o 2 estrellas. La siguiente tabla conserva el texto para inspeccionar ejemplos concretos. También se muestra una muestra aleatoria de 20 reseñas, suficiente para revisar manualmente el comportamiento sin presentar la inspección como una métrica estadística.

In [4]:
false_negatives = enriched[
    (enriched["rating"] >= 4) & (enriched["predicted_stars"] <= 2)
].copy()
print(f"Falsos negativos identificados: {len(false_negatives)}")
display(false_negatives[["review_id", "rating", "predicted_stars", "sentiment_band", "review_text"]].head(10))

manual_sample = enriched.sample(n=min(20, len(enriched)), random_state=42)
display(manual_sample[["review_id", "rating", "predicted_stars", "sentiment_band", "review_text"]])
print("Hipótesis de revisión: los desacuerdos suelen venir de contexto de servicio, ironía, quejas sobre esperas o elogios al personal que no se parecen a reseñas de productos.")

Falsos negativos identificados: 16


,review_id,rating,predicted_stars,sentiment_band,review_text
13,14,4,1,negative,Tried Harbor House Café after seeing it recomm...
14,15,5,1,negative,Every dish was bursting with flavor. The place...
72,73,5,1,negative,Stopped by Harbor House Café for the first tim...
86,87,5,1,negative,Tried Harbor House Café after seeing it recomm...
137,138,5,2,negative,Tried Harbor House Café after seeing it recomm...
199,200,5,2,negative,Stopped by Harbor House Café for the first tim...
203,204,5,1,negative,Came to Harbor House Café for a birthday brunc...
253,254,5,1,negative,Stopped by Harbor House Café for the first tim...
269,270,5,1,negative,Stopped by Harbor House Café for the first tim...
321,322,5,1,negative,Every dish was bursting with flavor. The staff...


,review_id,rating,predicted_stars,sentiment_band,review_text
361,362,3,3,neutral,Stopped by Harbor House Café for the first tim...
73,74,5,5,positive,Stopped by Harbor House Café for the first tim...
374,375,5,5,positive,Grabbed a quick coffee at Harbor House Café th...
155,156,4,4,positive,Grabbed a quick coffee at Harbor House Café th...
104,105,5,5,positive,Grabbed a quick coffee at Harbor House Café th...
394,395,4,4,positive,The food was absolutely delicious. We waited 2...
377,378,5,5,positive,Visited Harbor House Café last weekend. Loved ...
124,125,5,5,positive,Regular customer at Harbor House Café here. Th...
68,69,5,5,positive,Visited Harbor House Café last weekend. The pl...
450,451,3,3,neutral,Regular customer at Harbor House Café here. Av...


Hipótesis de revisión: los desacuerdos suelen venir de contexto de servicio, ironía, quejas sobre esperas o elogios al personal que no se parecen a reseñas de productos.


## Conclusiones para la account manager

El reporte debe presentar juntas la media de estrellas y la distribución textual. Una media de 4.5/5 indica satisfacción agregada, pero las bandas del modelo pueden descubrir neutralidad, críticas o desacuerdos que las estrellas ocultan. Los falsos negativos deben revisarse antes de automatizar decisiones: el desajuste entre reseñas de productos y servicios limita el uso del modelo como único criterio.

La salida operativa queda en `data/processed/reviews_with_sentiment.csv`, con la reseña original, la puntuación humana, las estrellas predichas, la confianza y la banda de sentimiento.